# 24. Unsupervised Learning: K-Means Clustering

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Low-Medium  
**Use Case**: Partition data into k clusters based on similarity

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the K-Means algorithm and its mechanics
- Implement K-Means clustering from scratch and using scikit-learn
- Determine optimal number of clusters (elbow method, silhouette score)
- Visualize clusters and centroids
- Handle initialization and convergence issues
- Apply K-Means to real-world problems

## Historical Context

K-Means was developed by Stuart Lloyd in 1957:
- Lloyd, S.P. (1957): "Least squares quantization in PCM"
- One of the most popular clustering algorithms
- Simple, efficient, and widely applicable

**Key Papers/References:**
- Lloyd, S.P. (1957). "Least squares quantization in PCM"
- MacQueen, J. (1967). "Some methods for classification and analysis of multivariate observations"

## When to Use K-Means Clustering

K-Means is appropriate when:
- You know or can estimate the number of clusters
- Clusters are spherical and similar in size
- Data is numerical and continuous
- You need fast, scalable clustering
- Clusters are well-separated
- Working with large datasets

## Theory & Mechanics

### Mathematical Foundation

K-Means minimizes the within-cluster sum of squares (WCSS):

**Objective Function:**
$$J = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

Where:
- $k$: Number of clusters
- $C_i$: Set of points in cluster $i$
- $\mu_i$: Centroid of cluster $i$

**Algorithm Steps:**

1. **Initialize**: Randomly select $k$ centroids
2. **Assign**: Assign each point to nearest centroid
3. **Update**: Recalculate centroids as mean of assigned points
4. **Repeat**: Steps 2-3 until convergence (centroids don't change)

**Convergence:**
- Algorithm converges when centroids stabilize
- Guaranteed to converge (but may find local optimum)
- Typically converges in few iterations

### How It Works

1. **Initialization**: Choose k initial centroids (random or k-means++)
2. **Assignment**: For each point, find nearest centroid
3. **Update**: Move centroids to mean of assigned points
4. **Check**: If centroids changed, go to step 2; else stop

### Key Hyperparameters

- **n_clusters (k)**: Number of clusters to form
- **init**: Initialization method ('k-means++', 'random', or array)
- **n_init**: Number of times to run with different centroids
- **max_iter**: Maximum iterations per run
- **tol**: Tolerance for convergence
- **random_state**: Seed for reproducibility

### Advantages

- Simple and easy to understand
- Fast and efficient (O(nk) per iteration)
- Scales well to large datasets
- Works well with spherical clusters
- Guaranteed convergence

### Limitations

- Requires specifying number of clusters
- Sensitive to initialization (local optima)
- Assumes spherical clusters
- Sensitive to outliers
- Doesn't work well with non-convex clusters
- All clusters assumed to have similar size


## Implementation

Let's implement K-Means clustering.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    make_blobs,  # Generate synthetic blob-shaped clusters (for demonstration)
    load_iris  # Iris flower dataset (for real-world example)
)
from sklearn.cluster import KMeans  # K-Means clustering algorithm
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.metrics import (
    silhouette_score,  # Calculate silhouette score (cluster quality metric)
    silhouette_samples  # Calculate silhouette score for each sample
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.unsupervised import (
    kmeans_cluster,  # K-Means clustering wrapper function
    evaluate_clustering  # Evaluate clustering quality (silhouette score)
)
from src.processing.preprocessing import scale_features  # Normalize features

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING SYNTHETIC DATASET: For Demonstration
# ============================================

# make_blobs() generates synthetic data with known cluster structure
# This is useful for understanding how K-Means works
# In real applications, you'd load your own data

# make_blobs() parameters:
# n_samples=300: Number of data points to generate
# centers=4: Number of clusters (blobs) to create
# n_features=2: Number of features (2D data for easy visualization)
# random_state=42: Ensures reproducible results
# cluster_std=0.60: Standard deviation of clusters (controls how spread out they are)
#   - Smaller = tighter clusters, larger = more spread out
X, y_true = make_blobs(
    n_samples=300,  # 300 data points
    centers=4,  # 4 clusters
    n_features=2,  # 2 features (2D data)
    random_state=42,  # Reproducibility
    cluster_std=0.60  # Cluster spread
)
# Returns:
# - X: Feature values (300 samples × 2 features)
# - y_true: True cluster labels (for comparison - we won't use these for clustering!)

print(f"Dataset Shape: {X.shape}")  # Output: (300, 2) - 300 points, 2 features
print(f"True number of clusters: {len(np.unique(y_true))}")  # Output: 4 clusters

# ============================================
# VISUALIZING ORIGINAL DATA: See the True Clusters
# ============================================

# Create figure with subplots
plt.figure(figsize=(10, 5))  # Width=10 inches, height=5 inches

# Subplot 1: True clusters (ground truth - what we're trying to discover)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot: X[:, 0] = first feature, X[:, 1] = second feature
# c=y_true colors points by true cluster labels
# cmap='viridis' is a color scheme
# s=50 sets point size
# alpha=0.7 makes points semi-transparent (easier to see overlapping points)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)

plt.title('True Clusters')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid (semi-transparent)

# Note: In real clustering, we don't know y_true!
# We're showing it here just to see what we're trying to discover

# ============================================
# APPLYING K-MEANS CLUSTERING
# ============================================

# Create KMeans model
# n_clusters=4: We know there are 4 clusters (in real applications, you'd need to find this)
# random_state=42: Ensures reproducible results
# n_init=10: Run algorithm 10 times with different initializations, keep best result
#   - K-Means can get stuck in local optima, so multiple runs help find better solution
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# fit_predict() trains the model and returns cluster assignments
# This is equivalent to: kmeans.fit(X) then kmeans.predict(X)
y_pred = kmeans.fit_predict(X)
# Returns: array of cluster labels (0, 1, 2, or 3 for each sample)

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Subplot 2: K-Means discovered clusters
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot of data points colored by predicted cluster
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_pred: Color by cluster assignment
# Same color = same cluster

# Plot cluster centers (centroids)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
# kmeans.cluster_centers_: Coordinates of cluster centers (centroids)
# [:, 0]: X-coordinates, [:, 1]: Y-coordinates
# marker='x': Show as X marks
# s=200: Large size
# c='red': Red color
# linewidths=3: Thick lines

plt.title('K-Means Clustering (k=4)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend (centroids)
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# DISPLAYING CLUSTERING METRICS
# ============================================

print(f"\nK-Means Results:")
print(f"  Number of clusters: {kmeans.n_clusters}")  # Number of clusters (4)

# Inertia (Within-Cluster Sum of Squares - WCSS)
# Measures how tight clusters are (sum of squared distances to centroids)
# Lower is better (tighter clusters)
print(f"  Inertia (WCSS): {kmeans.inertia_:.2f}")
# Lower inertia = better clustering (points closer to their centroids)

# Number of iterations until convergence
print(f"  Number of iterations: {kmeans.n_iter_}")  # How many iterations it took

# Interpretation:
# - Compare left plot (true clusters) with right plot (discovered clusters)
# - If they match well, K-Means found the right clusters
# - Centroids (red X's) are the "center" of each cluster
# - Lower inertia = better clustering quality


## Finding Optimal Number of Clusters

Let's use the elbow method and silhouette score to find the optimal k.


In [ ]:
# ============================================
# ELBOW METHOD: Finding Optimal Number of Clusters
# ============================================

# The "elbow method" helps find the optimal number of clusters (k)
# We test different k values and look for the "elbow" in the inertia plot
# The elbow is where adding more clusters doesn't significantly reduce inertia

# Test k values from 1 to 10
k_range = range(1, 11)  # [1, 2, 3, ..., 10]
inertias = []  # Store inertia (WCSS) for each k
silhouette_scores = []  # Store silhouette score for each k (alternative method)

# Test each k value
for k in k_range:
    # Create K-Means with this k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    
    # Train the model
    kmeans.fit(X)
    
    # Store inertia (WCSS) - lower is better
    inertias.append(kmeans.inertia_)
    # Inertia decreases as k increases (more clusters = tighter clusters)
    # But we want to find the "sweet spot" where adding more clusters doesn't help much
    
    # Calculate silhouette score (alternative method for finding optimal k)
    # Silhouette score measures how similar a point is to its own cluster vs other clusters
    # Range: -1 to 1 (higher is better)
    # Requires at least 2 clusters (can't calculate for k=1)
    if k > 1:  # Silhouette score requires at least 2 clusters
        # silhouette_score() calculates average silhouette score across all samples
        # X: Feature data, kmeans.labels_: Cluster assignments
        silhouette_avg = silhouette_score(X, kmeans.labels_)
        silhouette_scores.append(silhouette_avg)
    else:
        # For k=1, silhouette score is undefined (set to -1 as placeholder)
        silhouette_scores.append(-1)

# ============================================
# VISUALIZING ELBOW METHOD AND SILHOUETTE SCORES
# ============================================

# Create figure with 2 subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1 row, 2 columns

# Plot 1: Elbow Method
axes[0].plot(k_range, inertias, 'bo-')  # Blue circles connected by lines
axes[0].set_xlabel('Number of Clusters (k)')  # X-axis: k value
axes[0].set_ylabel('Inertia (WCSS)')  # Y-axis: within-cluster sum of squares
axes[0].set_title('Elbow Method')  # Chart title
axes[0].grid(True, alpha=0.3)  # Add grid
# Draw vertical line at optimal k (we know it's 4 from true labels)
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[0].legend()  # Show legend

# Plot 2: Silhouette Scores
axes[1].plot(k_range, silhouette_scores, 'ro-')  # Red circles connected by lines
axes[1].set_xlabel('Number of Clusters (k)')  # X-axis: k value
axes[1].set_ylabel('Silhouette Score')  # Y-axis: silhouette score (-1 to 1)
axes[1].set_title('Silhouette Score')  # Chart title
axes[1].grid(True, alpha=0.3)  # Add grid
# Draw vertical line at optimal k
axes[1].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[1].legend()  # Show legend

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# FINDING OPTIMAL K USING SILHOUETTE SCORE
# ============================================

# Find k with maximum silhouette score
# np.argmax() finds index of maximum value
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters (by silhouette): {optimal_k}")
print(f"  Silhouette score: {max(silhouette_scores):.3f}")

# Interpretation:
# - Elbow method: Look for "elbow" where inertia stops decreasing rapidly
# - Silhouette score: Higher is better (closer to 1 = better clustering)
# - Both methods should agree on optimal k
# - In this case, both should suggest k=4 (matching true number of clusters)


## Validation & Testing

Let's validate the clustering and compare different k values.


In [ ]:
# Evaluate clustering
evaluation = evaluate_clustering(X, y_pred, algorithm='KMeans')
print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
print(f"  Number of clusters: {evaluation['n_clusters']}")
print(f"  Inertia: {kmeans.inertia_:.2f}")

# Compare different k values
k_values = [2, 3, 4, 5, 6]
comparison_results = []

for k in k_values:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_test.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    comparison_results.append({
        'k': k,
        'inertia': kmeans_test.inertia_,
        'silhouette': sil_score
    })
    print(f"\nk={k}:")
    print(f"  Inertia: {kmeans_test.inertia_:.2f}")
    print(f"  Silhouette: {sil_score:.3f}")

# Assertions
assert kmeans.n_clusters == 4, "Expected 4 clusters"
assert kmeans.inertia_ > 0, "Inertia should be positive"
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
print("\n✓ Validation checks passed")


## Silhouette Analysis

Let's perform detailed silhouette analysis.


In [ ]:
# ============================================
# SILHOUETTE ANALYSIS: Detailed Cluster Quality Assessment
# ============================================

# Silhouette analysis shows how well each sample fits its assigned cluster
# This helps identify:
# - Well-clustered samples (high silhouette score)
# - Poorly-clustered samples (low or negative silhouette score)
# - Clusters with consistent quality

# Train final K-Means model with optimal k
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_final = kmeans_final.fit_predict(X)  # Get cluster assignments

# Calculate silhouette score for each sample (not just average)
# silhouette_samples() returns array with one score per sample
silhouette_vals = silhouette_samples(X, labels_final)
# Range: -1 to 1 (higher = better fit to assigned cluster)

# ============================================
# VISUALIZING SILHOUETTE SCORES
# ============================================

# Create figure for silhouette plot
fig, ax = plt.subplots(figsize=(10, 6))  # Figure size: 10×6 inches
y_lower = 10  # Starting y-position for first cluster

# Plot silhouette for each cluster
for i in range(4):  # Loop through 4 clusters
    # Get silhouette scores for samples in this cluster
    ith_cluster_silhouette_values = silhouette_vals[labels_final == i]
    # labels_final == i: Boolean mask (True for samples in cluster i)
    
    # Sort scores (for better visualization)
    ith_cluster_silhouette_values.sort()
    
    # Calculate size of this cluster
    size_cluster_i = ith_cluster_silhouette_values.shape[0]  # Number of samples
    
    # Calculate y-position for this cluster
    y_upper = y_lower + size_cluster_i  # End position
    
    # Choose color for this cluster
    color = plt.cm.viridis(float(i) / 4)  # Different color for each cluster
    
    # Draw horizontal bar for this cluster
    # fill_betweenx() fills area between two x-values
    # np.arange(y_lower, y_upper): Y-positions (one per sample)
    # 0: Start x-position (left edge)
    # ith_cluster_silhouette_values: End x-position (silhouette score)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                     facecolor=color, edgecolor=color, alpha=0.7)
    # facecolor: Fill color, edgecolor: Border color, alpha: Transparency
    
    # Add cluster label in the middle of the bar
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    # -0.05: X-position (left of y-axis), y_lower + 0.5 * size: Middle of cluster
    
    # Move to next cluster position (add spacing)
    y_lower = y_upper + 10  # 10 pixels spacing between clusters

# Label axes
ax.set_xlabel('Silhouette Coefficient Values')  # X-axis: silhouette score (-1 to 1)
ax.set_ylabel('Cluster Label')  # Y-axis: sample index (grouped by cluster)
ax.set_title('Silhouette Plot for K-Means (k=4)')  # Chart title

# Draw vertical line at average silhouette score
avg_silhouette = silhouette_score(X, labels_final)  # Average across all samples
ax.axvline(x=avg_silhouette, color="red", linestyle="--", 
           label=f'Average: {avg_silhouette:.3f}')
# Red dashed line shows overall average quality

ax.legend()  # Show legend
plt.tight_layout()  # Adjust layout
plt.show()  # Display the plot

# Interpretation:
# - Wide bars = good clustering (samples fit well)
# - Narrow bars = poor clustering (samples don't fit well)
# - Bars extending to left (negative) = samples may be in wrong cluster
# - Average line shows overall quality


## Real-World Application

Let's apply K-Means to the Iris dataset.


In [ ]:
# ============================================
# REAL-WORLD APPLICATION: Iris Dataset
# ============================================

# Apply K-Means to a real dataset (Iris flowers)
# This demonstrates how K-Means works on actual data

# Load Iris dataset
iris = load_iris()  # Returns Bunch object
X_iris = iris.data  # Features: flower measurements (150 samples × 4 features)
y_iris = iris.target  # True labels: species (0, 1, or 2) - for comparison only!

# ============================================
# FEATURE SCALING: Critical for K-Means
# ============================================

# K-Means uses distances, so features must be on similar scales
# StandardScaler normalizes features to mean=0, std=1
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)
# fit_transform() learns scaling from data and applies it

# ============================================
# APPLYING K-MEANS TO IRIS DATA
# ============================================

# Create K-Means with k=3 (Iris has 3 species)
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
# n_clusters=3: We know there are 3 species (in practice, you'd need to find this)

# Train and get cluster assignments
y_iris_pred = kmeans_iris.fit_predict(X_iris_scaled)
# Returns: array of cluster labels (0, 1, or 2 for each sample)

# ============================================
# EVALUATING CLUSTERING RESULTS
# ============================================

# Calculate silhouette score (measures clustering quality)
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
# Range: -1 to 1 (higher = better clustering)

print("Iris Dataset Clustering:")
print(f"  Number of clusters: {kmeans_iris.n_clusters}")  # 3 clusters
print(f"  Silhouette Score: {sil_score_iris:.3f}")  # Quality metric
print(f"  Inertia: {kmeans_iris.inertia_:.2f}")  # WCSS (lower is better)

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Create figure with 2 subplots (comparing true labels vs discovered clusters)
plt.figure(figsize=(12, 5))  # Width=12 inches, height=5 inches

# Subplot 1: True labels (ground truth - what we're trying to discover)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot using first 2 features (for 2D visualization)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
# X_iris[:, 0]: First feature (sepal length)
# X_iris[:, 1]: Second feature (sepal width)
# c=y_iris: Color by true species labels
# cmap='viridis': Color scheme
# s=50: Point size
# alpha=0.7: Semi-transparent

# Plot centroids (if we had them for true labels - this is just for comparison)
# Note: We're plotting K-Means centroids here for visual consistency
plt.scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
# Red X marks show cluster centers

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('True Labels')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: K-Means discovered clusters
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot colored by predicted clusters
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_iris_pred: Color by predicted cluster assignment

# Plot cluster centers (centroids)
plt.scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
# Red X marks show discovered cluster centers

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('K-Means Clustering (k=3)')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation:
# - Compare left plot (true species) with right plot (discovered clusters)
# - If colors match well, K-Means found the right clusters
# - Note: We're only using 2 features for visualization (4D data reduced to 2D)
# - In reality, K-Means uses all 4 features for clustering


## Summary & Key Takeaways

### Key Concepts Learned

1. **K-Means Basics**
   - Partition-based clustering algorithm
   - Minimizes within-cluster sum of squares
   - Requires specifying number of clusters
   - Iterative algorithm (assign, update, repeat)

2. **Algorithm Steps**
   - Initialize k centroids
   - Assign points to nearest centroid
   - Update centroids to mean of assigned points
   - Repeat until convergence

3. **Finding Optimal k**
   - **Elbow Method**: Plot inertia vs k, look for "elbow"
   - **Silhouette Score**: Measure cluster quality (-1 to 1, higher is better)
   - **Domain Knowledge**: Use prior knowledge about data

4. **Best Practices**
   - Scale features before clustering
   - Use k-means++ initialization (default)
   - Run multiple times with different seeds
   - Visualize results to validate clusters
   - Use silhouette analysis for validation

### When to Use K-Means

✅ **Good for:**
- Known or estimable number of clusters
- Spherical, well-separated clusters
- Numerical, continuous data
- Large datasets (scalable)
- When speed is important
- Exploratory data analysis

❌ **Not ideal for:**
- Unknown number of clusters
- Non-spherical clusters
- Clusters of different sizes
- Categorical data
- Outliers (sensitive)
- Non-convex cluster shapes

### Next Steps

- Try **K-Means++** initialization for better results
- Explore **Mini-Batch K-Means** for very large datasets
- Compare with **Hierarchical Clustering** for different cluster shapes
- Use **DBSCAN** for density-based clustering
- Apply **PCA** before clustering for high-dimensional data
